In [10]:
import torch
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split
import torch.nn as nn
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch.optim as optim
import torchmetrics as tm

# NN.Module / NN.Sequenital

* nn.Module is PyTorch's base class for all neural network components. Every model, every layer, and every reusable building block in PyTorch inherits from it. It is not a model itself; it is the infrastructure that makes models work.

* Mainly in examples i've seen so far, the model was being written as a class and the rest was kept with object instances

* nn.Sequential is itself an nn.Module subclass. It is sufficient when layers are applied in a fixed, linear order with no custom logic.

Write a custom class when:

The forward pass requires branching, skip connections, or intermediate outputs
The model has multiple sub-networks (e.g. encoder + decoder) that need to be held as separate attributes
You want to pass hyperparameters (e.g. hidden_size) at instantiation time

```
# Sequential is enough
model = nn.Sequential(nn.Linear(9, 64), nn.ReLU(), nn.Linear(64, 1))

# Custom class needed (skip connection example)
class ResidualBlock(nn.Module):
    def forward(self, x):
        return x + self.net(x)   # cannot express this in Sequential
```

## two parts:

* __init__(self) -- define the layers and sub-modules here. Call super().__init__() first so that nn.Module can initialise its own internal state. Any layer assigned to self. is automatically registered and tracked.

* forward(self, x) -- define what happens during the forward pass. This is where you specify how data flows through the layers you defined in __init__. PyTorch calls this method when you do model(x).

```
class Net(nn.Module):
    def __init__(self):
        super().__init__()          # required: initialises nn.Module internals
        self.fc1 = nn.Linear(9, 16) # registered automatically
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):           # required: defines the computation
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

net = Net()
```

* You never call forward() directly. Calling net(x) triggers it internally via Python's __call__ mechanism, which also runs hooks and other PyTorch bookkeeping around it.

# Device Management (Using GPU/CUDA)

# TensorDataset vs Custom Dataset Class

* Use TensorDataset when your data is already in memory as tensors or numpy arrays and requires no per-sample processing.

```
pythondataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)
```

#### Write a custom Dataset class when:

Data lives on disk and *must be loaded lazily* (images, large CSVs)
Each sample requires a transform or augmentation at load time
Multiple source files need to be joined into a single sample

```
class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        df = pd.read_csv(csv_path)
        self.data = df.to_numpy()

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        label    = self.data[idx, -1]
        return features, label
```

__getitem__ and __len__ are standart python dunder(double underscore) methods.

* getitem:

* take one argument called idx return features and label for a single
sample at index idx

# Implementing a Custom Dataset Class with lazy loading  parquet data
https://discuss.pytorch.org/t/loading-big-parquet-files/11990/5\




In [11]:
# we delegated reading the data with pandasand creating a tensor set/numpy array to dataset
class HomeDataset(Dataset):

    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        self.data = df.to_numpy()

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        features = self.data[idx, -1]

In [7]:
class HomeLazyDatasetWithoutCache(Dataset):

    def __init__(self, parquet_path):
        self.parquet_path = parquet_path

        self.pf = pq.ParquetFile(parquet_path) #doesnt load the entire parquet to the RAM!

        metadata = self.pf.metadata
        self.num_rows = metadata.num_rows
        self.num_row_groups = metadata.num_row_groups

        # parquet has row groups,last one might have different size
        self.row_group_sizes = [
            metadata.row_group(i).num_rows
            for i in range(self.num_row_groups)
        ]

        # start index of each row group
        # row_group_sizes = [1000, 1000, 623]
        # [0] + sizes[:-1] = [0, 1000, 1000]  → shift sizes right, prepend 0
        # cumsum([0, 1000, 1000]) = [0, 1000, 2000] → start index of each row group
        self.row_group_offsets = np.cumsum([0] + self.row_group_sizes[:-1])

    def __len__(self):
        return self.num_rows

    def _get_row_group_and_local_idx(self, idx):
        # finding out which row group has the wanted id
        group_idx = np.searchsorted(self.row_group_offsets, idx, side="right") - 1

        # rows index at its row group
        local_idx = idx - self.row_group_offsets[group_idx]
        return int(group_idx), int(local_idx)

    def __getitem__(self, idx):
        group_idx, local_idx = self._get_row_group_and_local_idx(idx)

        # reading just that row group
        table = self.pf.read_row_group(group_idx)
        row = table.to_pandas().iloc[local_idx].to_numpy(dtype=np.float32)

        features = row[:-1]
        label = row[-1]

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )

## Fixing Lazy Loadings Train Test Split Problem
Lazy loading implemented here might cause troubles (loading same row group again and again) as the indexes are picked randomly. Introducing caching and keeping id of last read group (which is already at the memory) would solve it.

Another concern I also had was, whether after splitting would the new datasets keep the data inside it. No, random_split returns Subset object with:

```
class Subset(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset      # orijinal dataset'e referans
        self.indices = indices      # hangi global indekslerin bu subset'e ait olduğu

    def __getitem__(self, idx):
        return self.dataset[self.indices[idx]]
```

They just carry indices and a reference to the original dataset


In [ ]:
class HomeLazyDataset(Dataset):

    def __init__(self, parquet_path):
        self.parquet_path = parquet_path

        self.pf = pq.ParquetFile(parquet_path)

        metadata = self.pf.metadata
        self.num_rows = metadata.num_rows
        self.num_row_groups = metadata.num_row_groups

        self.row_group_sizes = [
            metadata.row_group(i).num_rows
            for i in range(self.num_row_groups)
        ]

        self.row_group_offsets = np.cumsum([0] + self.row_group_sizes[:-1])

        # cache for the last read row group
        self._cached_group_idx = -1
        self._cached_table = None

    def __len__(self):
        return self.num_rows

    def _get_row_group_and_local_idx(self, idx):
        group_idx = np.searchsorted(self.row_group_offsets, idx, side="right") - 1
        local_idx = idx - self.row_group_offsets[group_idx]
        return int(group_idx), int(local_idx)

    def __getitem__(self, idx):
        group_idx, local_idx = self._get_row_group_and_local_idx(idx)

        if group_idx != self._cached_group_idx:
            self._cached_table = self.pf.read_row_group(group_idx).to_pandas()
            self._cached_group_idx = group_idx

        row = self._cached_table.iloc[local_idx].to_numpy(dtype=np.float32)

        features = row[:-1]
        label = row[-1]

        return (
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )

In [13]:
path = "./data/house_prices.parquet"

home_dataset = HomeLazyDataset(path)

batch_size = 2

total_len = len(lazy_home_dataset)
n_train = int(0.7 * total_len)
n_val = int(0.15 * total_len)
n_test = total_len - n_train - n_val

train_set, val_set, test_set = random_split(lazy_home_dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)


#  Common: Implementing Custom Net (Neural Network)

Helpers introduced here: nn.functional.relu, activation functions in functional formats

In [15]:
# both code implements same network structure

# sequential model def:
net = nn.Sequential(
    nn.Linear(9, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),

)

# class based model definition:
class Net(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(9, 16) #FC = Fully connected
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)


    # first fc1 applied on initial X, then relu.
    # then fc2 applied, again relu
    # finally fc3 and sigmoid
    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()



In [ ]:
epochs = 1000

criterion = nn.MSELoss()
optimizer = optim.SGD(net.parameters(), lr=0.01)

for epoch in range(epochs):
    net.train()
    for features, labels in train_loader:
        optimizer.zero_grad()
        outputs = net(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    net.eval()
    with torch.no_grad():
        for features, labels in val_loader:
            outputs = net(features)

